# Traffic Accidents Analysis
Analysis of `traffic_accidents_cleaned.csv` for Issue #4.

In [ ]:
import pandas as pd
from pathlib import Path

INPUT = Path('traffic_accidents_cleaned.csv')
df = pd.read_csv(INPUT)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month_name()
df['Month_Num'] = df['Date'].dt.month
df['Weekday'] = df['Date'].dt.day_name()
df['Hour'] = df['Time'].dt.hour

def time_block(h):
    if h < 6: return '00:00-05:59'
    if h < 12: return '06:00-11:59'
    if h < 18: return '12:00-17:59'
    return '18:00-23:59'

df['Time_Block'] = df['Hour'].apply(time_block)
print('Rows:', len(df), '| Columns:', len(df.columns))
df.head()


: 

## 1. Basic statistics

In [ ]:
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Cities:', df['City'].nunique(), '| States:', df['State'].nunique())
display(df[['Vehicles_Involved','Injuries','Fatalities']].describe().T)
print('Total vehicles involved:', df['Vehicles_Involved'].sum())
print('Total injuries:', df['Injuries'].sum())
print('Total fatalities:', df['Fatalities'].sum())


## 2. Year, month, weekday and time analysis

In [ ]:
year_counts = df['Year'].value_counts().sort_index()
month_counts = df.groupby(['Month_Num','Month']).size().sort_index()
weekday_counts = df['Weekday'].value_counts()
time_counts = df['Time_Block'].value_counts()
display(year_counts.to_frame('Accidents'))
display(month_counts.to_frame('Accidents'))
display(weekday_counts.to_frame('Accidents'))
display(time_counts.to_frame('Accidents'))


## 3. Severity, location, weather, road, vehicle and cause analysis

In [ ]:
for col in ['Accident_Severity','City','Weather_Condition','Road_Condition','Vehicle_Type','Accident_Cause']:
    print('\n', col)
    display(df[col].value_counts().to_frame('Accidents'))


## 4. Relationships

In [ ]:
severity_weather = pd.crosstab(df['Weather_Condition'], df['Accident_Severity'])
severity_cause = pd.crosstab(df['Accident_Cause'], df['Accident_Severity'])
display(severity_weather)
display(severity_cause)


## 5. Key findings
- 500 accidents are recorded across 2023–2024, with 250 in each year.
- The 00:00–05:59 block has the highest accident count (137).
- July has the highest monthly count (50); June has the lowest (29).
- Minor accidents are most common (244), followed by Moderate (174).
- Delhi has the highest city count (36).
- Rain is the most common weather category (110), and Poor is the most common road condition (108).
- Bus is the most frequent vehicle type (80).
- Speeding is the most frequently recorded accident cause (79).

These are descriptive associations in the dataset, not causal conclusions.


In [ ]:
results = []
def add_counts(section, series):
    counts = series.value_counts()
    for category, count in counts.items():
        results.append({'section':section,'metric':'accidents','category':category,
                        'count':int(count),'percentage':round(count/len(df)*100,1)})

results += [
 {'section':'overall','metric':'total_accidents','category':'All','count':len(df),'percentage':100.0},
 {'section':'overall','metric':'total_vehicles_involved','category':'All','count':int(df['Vehicles_Involved'].sum()),'percentage':''},
 {'section':'overall','metric':'total_injuries','category':'All','count':int(df['Injuries'].sum()),'percentage':''},
 {'section':'overall','metric':'total_fatalities','category':'All','count':int(df['Fatalities'].sum()),'percentage':''},
]
for section, series in [('year',df['Year']),('month',df['Month']),('time_block',df['Time_Block']),('weekday',df['Weekday']),('severity',df['Accident_Severity']),('city',df['City']),('weather',df['Weather_Condition']),('road',df['Road_Condition']),('vehicle',df['Vehicle_Type']),('cause',df['Accident_Cause'])]:
    add_counts(section, series)
pd.DataFrame(results).to_csv('analysis_results.csv', index=False)
print('Saved analysis_results.csv')
